# B9. Render the attacker search grid on Kaggle

Renders the 50 tuning photos, C-on and I-on, under 36 settings (font size 16, 24, 32 x contrast high, low x font DejaVu, Comic Neue x text written 1, 2, 3 times), then applies blur sigma 2, JPEG 20 and OCR-inpaint. That is 10,800 defended images, the rows of `manifests/attacker_search.csv`.

**Before you start**
1. Settings: Accelerator **GPU T4** (EasyOCR needs it, on CPU this takes about 6 hours), Internet **on**.
2. Add-ons > Secrets: `GITHUB_TOKEN` switched on.

**How to run**
1. Click **Save Version > Save & Run All (Commit)**. Not Run All, or the files are not kept. About 45 minutes.
2. When it finishes, the version's **Output** tab should list `images_attacker_01.zip` to `images_attacker_05.zip`, `ocr_masks_attacker.zip` and `attacker_search.csv`, and the last cell should print `OK`.
3. On the Output tab use **New Dataset**, call it `pixelguard-attacker`, then **Share** it with Mufeed.

In [ ]:
!pip install -q easyocr
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, OCR will be very slow, turn on the GPU')

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

os.chdir('/kaggle/working')
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = 'github.com/Mehdi-Abidi/VisionLanguageTypographicDefence.git'
subprocess.run(['rm', '-rf', '/tmp/repo'])
r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{repo}', '/tmp/repo'], capture_output=True, text=True)
print('cloned' if r.returncode == 0 else 'clone failed: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', '/tmp/repo', 'remote', 'set-url', 'origin', f'https://{repo}'])
del token
os.chdir('/tmp/repo')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
import time
t0 = time.time()
!python src/make_attacker_set.py --images --out /tmp/b9 --cache /tmp/coco
print(f'took {(time.time() - t0) / 60:.0f} min')

def same_as_git(f):
    blob = subprocess.run(['git', 'show', f'HEAD:{f}'], capture_output=True).stdout
    return blob.replace(b'\r\n', b'\n') == open(f, 'rb').read().replace(b'\r\n', b'\n')
same = same_as_git('manifests/attacker_search.csv')
print('manifest the same as the committed one:', same)

In [ ]:
import math, shutil, zipfile
from pathlib import Path
import pandas as pd

manifest = pd.read_csv('manifests/attacker_search.csv')
missing = [p for p in manifest['path'] if not (Path('/tmp/b9') / p).exists()]
masks = sorted(Path('/tmp/b9/images').glob('*_mask.png'))
with zipfile.ZipFile('/kaggle/working/ocr_masks_attacker.zip', 'w') as z:
    for m in masks:
        z.write(m, m.name)
ids = list(dict.fromkeys(manifest['image_id'].astype(str)))
size = math.ceil(len(ids) / 5)
packed = 0
for k in range(5):
    chunk = set(ids[k * size:(k + 1) * size])
    part = manifest[manifest['image_id'].astype(str).isin(chunk)]
    with zipfile.ZipFile(f'/kaggle/working/images_attacker_{k + 1:02d}.zip', 'w', zipfile.ZIP_STORED) as z:
        for p in part['path']:
            z.write(Path('/tmp/b9') / p, p)
            packed += 1
shutil.copy('manifests/attacker_search.csv', '/kaggle/working/attacker_search.csv')
print(len(manifest), 'manifest rows,', len(missing), 'missing,', packed, 'packed,', len(masks), 'OCR masks')
print('output:', {p.name: f'{p.stat().st_size / 1e6:.0f} MB' for p in sorted(Path('/kaggle/working').iterdir()) if p.is_file()})
print('OK' if not missing and same and packed == len(manifest) else 'PROBLEM')